# Whisper STT Implementation Lab

This notebook processes `TX048clip.mp3` with the OpenAI Whisper API.

In [4]:
# Import Path for reliable file and folder handling.
from pathlib import Path

# Import os for reading environment variables such as the API key.
import os

# Import shutil so we can locate the FFmpeg executable.
import shutil

# Import load_dotenv to load the API key from a local .env file.
from dotenv import load_dotenv

# Import the OpenAI client.
from openai import OpenAI


# Use the folder containing the notebook as the project folder.
PROJECT_ROOT = Path.cwd()

# Define the provided audio file.
AUDIO_FILE = PROJECT_ROOT / "TX048clip.mp3"

# Define the folder for temporary audio chunks.
CHUNKS_DIR = PROJECT_ROOT / "audio_chunks"

# Define the folder for final transcription files.
OUTPUT_DIR = PROJECT_ROOT / "transcription_outputs"


# Create the required folders if they do not already exist.
CHUNKS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)


# Load the API key from a local .env file.
load_dotenv()

# Read the API key without displaying it.
api_key = os.getenv("OPENAI_API_KEY")

# Stop with a clear message if the API key is missing.
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is missing. Add it to your local .env file."
    )


# Create the OpenAI API client.
client = OpenAI(api_key=api_key)


# Find FFmpeg on the system PATH.
FFMPEG_PATH = shutil.which("ffmpeg")

# Use the Anaconda FFmpeg location if it is not on PATH.
if FFMPEG_PATH is None:
    fallback_ffmpeg = Path("/opt/anaconda3/bin/ffmpeg")

    if fallback_ffmpeg.exists():
        FFMPEG_PATH = str(fallback_ffmpeg)


# Stop if FFmpeg cannot be found.
if FFMPEG_PATH is None:
    raise FileNotFoundError(
        "FFmpeg was not found. Install it before continuing."
    )


# Stop if the provided audio file is missing.
if not AUDIO_FILE.exists():
    raise FileNotFoundError(
        f"Audio file not found: {AUDIO_FILE}"
    )


# Print safe setup information.
# The API key itself is never printed.
print("Environment setup complete.")
print(f"Python project folder: {PROJECT_ROOT}")
print(f"Audio file: {AUDIO_FILE}")
print(f"Audio file size: {AUDIO_FILE.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Chunks folder: {CHUNKS_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"OpenAI client initialized: {client is not None}")
print(f"FFmpeg path: {FFMPEG_PATH}")

Environment setup complete.
Python project folder: /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation
Audio file: /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation/TX048clip.mp3
Audio file size: 1.17 MB
Chunks folder: /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation/audio_chunks
Output folder: /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation/transcription_outputs
OpenAI client initialized: True
FFmpeg path: /opt/anaconda3/bin/ffmpeg


In [5]:
# Import tools for running FFprobe and reading its JSON response.
import json
import subprocess


# FFprobe is installed alongside FFmpeg.
# We derive its expected path from the FFmpeg path.
ffprobe_path = Path(FFMPEG_PATH).with_name("ffprobe")


# Stop if FFprobe cannot be found.
if not ffprobe_path.exists():
    raise FileNotFoundError(
        f"FFprobe was not found at: {ffprobe_path}"
    )


# Ask FFprobe for the audio format, duration, codec, sample rate, and channels.
ffprobe_command = [
    str(ffprobe_path),
    "-v", "error",
    "-show_entries",
    "format=duration,format_name,size:stream=codec_name,sample_rate,channels",
    "-of", "json",
    str(AUDIO_FILE),
]


# Execute FFprobe and capture its output.
probe_result = subprocess.run(
    ffprobe_command,
    capture_output=True,
    text=True,
    check=True,
)


# Convert FFprobe's JSON text into a Python dictionary.
audio_info = json.loads(probe_result.stdout)


# Extract the file-level information.
format_info = audio_info.get("format", {})

# Extract information about the first audio stream.
stream_info = audio_info.get("streams", [{}])[0]


# Convert the duration from text to seconds.
duration_seconds = float(format_info.get("duration", 0))


# Print the audio properties.
print("Audio inspection complete.")
print(f"Format: {format_info.get('format_name')}")
print(f"Duration: {duration_seconds:.2f} seconds")
print(f"Codec: {stream_info.get('codec_name')}")
print(f"Sample rate: {stream_info.get('sample_rate')} Hz")
print(f"Channels: {stream_info.get('channels')}")
print(f"Size: {int(format_info.get('size', 0)) / (1024 * 1024):.2f} MB")

Audio inspection complete.
Format: mp3
Duration: 104.39 seconds
Codec: mp3
Sample rate: 44100 Hz
Channels: 1
Size: 1.17 MB


In [6]:
# Open the audio file in binary-read mode.
# The API requires the audio file bytes.
with AUDIO_FILE.open("rb") as audio_stream:

    # Send the complete audio file to the Whisper transcription API.
    basic_transcript_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_stream,
    )


# Extract the plain transcription text from the API response.
basic_transcript_text = basic_transcript_response.text


# Display the basic transcription.
print("Basic transcription complete.")
print("-" * 60)
print(basic_transcript_text)

Basic transcription complete.
------------------------------------------------------------
CEDAS has entered Customs, and the next step is to prepare a Mexican Customs entry. In Mexico we call them Pedimentos. It's really an appraisal sheet. It's a breakdown of the sheet. Usually the same information. That's right. But here you give them all the information, for example, let me see if I've got one right here. You give them all the information on what the invoice contains. There's one. On what the invoice contains, what the material is. It's all physically here. For example, this is some bottling equipment going over there. And on this appraisal sheet you tell them who it's for. Who made the invoice out of the States. This came from St. Louis, Missouri. How much the freight was. And on the reverse of this thing, you put that everything has to be marked. In other words, this identification number is that traffic number I spoke to you a little while ago. Number of boxes. What they contain

In [7]:
# Provide domain-specific vocabulary that may help Whisper recognize
# names, customs terminology, and technical phrases correctly.
transcription_prompt = (
    "The meeting discusses Mexican customs documentation and international freight. "
    "Important terms include CEDAS, Customs, Mexican Customs entry, pedimentos, "
    "appraisal sheet, invoice, bottling equipment, St. Louis, Missouri, freight, "
    "identification number, traffic number, tariff duty, tariff number, "
    "ad valorem rate, duties, appraiser, and Vistas."
)


# Reopen the audio file from the beginning.
# A new file handle ensures the API receives the complete audio.
with AUDIO_FILE.open("rb") as audio_stream:

    # Transcribe the audio with the domain-specific prompt.
    prompted_transcript_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_stream,
        prompt=transcription_prompt,
    )


# Extract the prompted transcription text.
prompted_transcript_text = prompted_transcript_response.text


# Display the prompted transcription.
print("Prompted transcription complete.")
print("-" * 60)
print(prompted_transcript_text)

Prompted transcription complete.
------------------------------------------------------------
CEDAS has entered Customs and the next step is to prepare a Mexican Customs entry, in Mexico we call them pedimentos. It's really an appraisal sheet, it's a breakdown of the sheet. Usually the same information. That's right. But here you give them all the information, for example, let me see, I've got one right here. You give them all the information on what the invoice contains, what the material is. Here, for example, is some bottling equipment going over there, and on this appraisal sheet you tell them who it's for, who made the invoice out of the state. This came from St. Louis, Missouri, how much the freight was, and on the reverse of this thing, everything has to be marked. Complete identification. This identification number is that traffic number I spoke to you a little while ago. Number of boxes, what they contain, the tariff duty, the tariff number, the ad valorem rate, and what the a

In [8]:
# Import SequenceMatcher to calculate a simple similarity score
# between the unprompted and prompted transcriptions.
from difflib import SequenceMatcher


# Calculate similarity between both transcription versions.
similarity_score = SequenceMatcher(
    None,
    basic_transcript_text.lower(),
    prompted_transcript_text.lower(),
).ratio()


# Define important domain terms from the audio.
important_terms = [
    "CEDAS",
    "pedimentos",
    "appraisal sheet",
    "bottling equipment",
    "St. Louis",
    "tariff",
    "ad valorem",
    "Vistas",
]


# Check which important terms appear in each transcription.
print("Prompt comparison")
print("-" * 60)
print(f"Similarity score: {similarity_score:.2%}")
print()


print("Term recognition:")
for term in important_terms:
    found_basic = term.lower() in basic_transcript_text.lower()
    found_prompted = term.lower() in prompted_transcript_text.lower()

    print(
        f"{term}: "
        f"unprompted={found_basic}, "
        f"prompted={found_prompted}"
    )


# Display both complete versions for qualitative comparison.
print("\nUNPROMPTED TRANSCRIPTION")
print("-" * 60)
print(basic_transcript_text)


print("\nPROMPTED TRANSCRIPTION")
print("-" * 60)
print(prompted_transcript_text)

Prompt comparison
------------------------------------------------------------
Similarity score: 47.07%

Term recognition:
CEDAS: unprompted=True, prompted=True
pedimentos: unprompted=True, prompted=True
appraisal sheet: unprompted=True, prompted=True
bottling equipment: unprompted=True, prompted=True
St. Louis: unprompted=True, prompted=True
tariff: unprompted=True, prompted=True
ad valorem: unprompted=True, prompted=True
Vistas: unprompted=True, prompted=True

UNPROMPTED TRANSCRIPTION
------------------------------------------------------------
CEDAS has entered Customs, and the next step is to prepare a Mexican Customs entry. In Mexico we call them Pedimentos. It's really an appraisal sheet. It's a breakdown of the sheet. Usually the same information. That's right. But here you give them all the information, for example, let me see if I've got one right here. You give them all the information on what the invoice contains. There's one. On what the invoice contains, what the material 

In [9]:
# Import tools for running FFmpeg and handling chunk file paths.
import subprocess


# Set the length of each chunk in seconds.
# A production workflow could use longer chunks, such as 10 minutes.
CHUNK_LENGTH_SECONDS = 60


# Store chunks in a separate folder for this chunk size.
CURRENT_CHUNKS_DIR = CHUNKS_DIR / f"{CHUNK_LENGTH_SECONDS}_second_chunks"

# Create the chunk folder if it does not already exist.
CURRENT_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)


# Define the output filename pattern.
# FFmpeg will create files such as chunk_000.mp3, chunk_001.mp3, and so on.
chunk_pattern = CURRENT_CHUNKS_DIR / "chunk_%03d.mp3"


# Build the FFmpeg command.
ffmpeg_chunk_command = [
    FFMPEG_PATH,
    "-y",
    "-i", str(AUDIO_FILE),
    "-map", "0:a:0",
    "-f", "segment",
    "-segment_time", str(CHUNK_LENGTH_SECONDS),
    "-reset_timestamps", "1",
    "-c:a", "libmp3lame",
    "-q:a", "4",
    str(chunk_pattern),
]


# Run FFmpeg to split the original audio.
subprocess.run(
    ffmpeg_chunk_command,
    check=True,
    capture_output=True,
    text=True,
)


# Collect the generated chunk files in order.
chunk_files = sorted(CURRENT_CHUNKS_DIR.glob("chunk_*.mp3"))


# Confirm that chunking produced files.
if not chunk_files:
    raise RuntimeError("FFmpeg did not create any audio chunks.")


# Display the chunking result.
print("Audio chunking complete.")
print(f"Chunk duration: {CHUNK_LENGTH_SECONDS} seconds")
print(f"Number of chunks: {len(chunk_files)}")

for chunk_file in chunk_files:
    print(f"- {chunk_file.name}")

Audio chunking complete.
Chunk duration: 60 seconds
Number of chunks: 2
- chunk_000.mp3
- chunk_001.mp3


In [10]:
# Store all timestamped segments from every audio chunk.
chunked_segments = []

# Store the complete text from every chunk.
chunked_text_parts = []


# Process each chunk in chronological order.
for chunk_index, chunk_file in enumerate(chunk_files):

    # Calculate when this chunk begins in the original audio.
    chunk_offset_seconds = chunk_index * CHUNK_LENGTH_SECONDS

    print(f"Transcribing {chunk_file.name}...")

    # Open the current chunk in binary-read mode.
    with chunk_file.open("rb") as audio_stream:

        # Request detailed transcription data with segment timestamps.
        chunk_response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_stream,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )

    # Save the chunk's full text.
    chunked_text_parts.append(chunk_response.text)

    # Read the timestamped segments returned by Whisper.
    response_segments = getattr(chunk_response, "segments", None) or []

    # Adjust each segment's local timestamps to global timestamps.
    for segment in response_segments:

        # Read the segment's timestamps relative to the current chunk.
        local_start = float(segment.start)
        local_end = float(segment.end)

        # Add the chunk offset to obtain timestamps for the original audio.
        global_start = chunk_offset_seconds + local_start
        global_end = chunk_offset_seconds + local_end

        # Store the corrected segment.
        chunked_segments.append(
            {
                "start": global_start,
                "end": global_end,
                "text": segment.text.strip(),
                "chunk": chunk_file.name,
            }
        )

    print(f"  Segments found: {len(response_segments)}")


# Combine all chunk text into one complete transcription.
chunked_transcript_text = " ".join(chunked_text_parts)


# Confirm the chunked transcription result.
print("\nChunked transcription complete.")
print(f"Total timestamped segments: {len(chunked_segments)}")
print("-" * 60)

for segment in chunked_segments:
    print(
        f"[{segment['start']:.2f}s - {segment['end']:.2f}s] "
        f"{segment['text']}"
    )

Transcribing chunk_000.mp3...
  Segments found: 18
Transcribing chunk_001.mp3...
  Segments found: 7

Chunked transcription complete.
Total timestamped segments: 25
------------------------------------------------------------
[0.00s - 9.00s] CEDAS has entered Customs and the next step is to prepare a Mexican Customs entry.
[9.00s - 13.00s] In Mexico we call them Pedimentos.
[13.00s - 19.00s] It's really an appraisal sheet. It's a breakdown of the sheet.
[19.00s - 20.00s] Usually the same information.
[20.00s - 21.00s] That's right.
[21.00s - 23.00s] But here you give them all the information.
[23.00s - 26.00s] For example, let me see if I've got one right here.
[26.00s - 33.00s] You give them all the information on...
[33.00s - 36.00s] on what the invoice contains.
[36.00s - 37.00s] There's one.
[37.00s - 39.00s] On what the invoice contains.
[39.00s - 43.00s] What the material is.
[43.00s - 44.00s] It's all physically here.
[44.00s - 48.00s] For example, this is some bottling equipmen

In [11]:
# Import JSON support for the structured export.
import json


# Convert seconds into an SRT-compatible timestamp.
# SRT timestamps use the format HH:MM:SS,mmm.
def seconds_to_srt_timestamp(seconds):
    # Convert the total time into whole seconds and milliseconds.
    total_milliseconds = round(seconds * 1000)

    # Calculate hours, minutes, seconds, and milliseconds.
    hours, remainder = divmod(total_milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    whole_seconds, milliseconds = divmod(remainder, 1_000)

    # Return the formatted SRT timestamp.
    return (
        f"{hours:02d}:{minutes:02d}:{whole_seconds:02d},"
        f"{milliseconds:03d}"
    )


# Create a human-readable TXT export.
txt_path = OUTPUT_DIR / "transcription_with_timestamps.txt"

with txt_path.open("w", encoding="utf-8") as text_file:

    # Write a short metadata heading.
    text_file.write("Whisper transcription with timestamps\n")
    text_file.write(f"Source file: {AUDIO_FILE.name}\n")
    text_file.write("=" * 60 + "\n\n")

    # Write every timestamped segment.
    for segment in chunked_segments:
        text_file.write(
            f"[{segment['start']:.2f}s - {segment['end']:.2f}s] "
            f"{segment['text']}\n"
        )


# Create an SRT subtitle export.
srt_path = OUTPUT_DIR / "transcription_with_timestamps.srt"

with srt_path.open("w", encoding="utf-8") as subtitle_file:

    # SRT entries start counting at 1.
    for subtitle_number, segment in enumerate(chunked_segments, start=1):

        # Convert the segment times into SRT timestamps.
        start_time = seconds_to_srt_timestamp(segment["start"])
        end_time = seconds_to_srt_timestamp(segment["end"])

        # Write one complete subtitle entry.
        subtitle_file.write(f"{subtitle_number}\n")
        subtitle_file.write(f"{start_time} --> {end_time}\n")
        subtitle_file.write(f"{segment['text']}\n\n")


# Create a JSON export containing metadata and timestamped segments.
json_path = OUTPUT_DIR / "transcription_with_timestamps.json"

json_data = {
    "source_file": AUDIO_FILE.name,
    "duration_seconds": duration_seconds,
    "chunk_length_seconds": CHUNK_LENGTH_SECONDS,
    "transcription": chunked_transcript_text,
    "segments": chunked_segments,
}

with json_path.open("w", encoding="utf-8") as json_file:

    # Write readable, indented JSON.
    json.dump(json_data, json_file, indent=2, ensure_ascii=False)


# Confirm that all export files were created.
print("Exports complete.")
print(f"TXT:  {txt_path}")
print(f"SRT:  {srt_path}")
print(f"JSON: {json_path}")

Exports complete.
TXT:  /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation/transcription_outputs/transcription_with_timestamps.txt
SRT:  /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation/transcription_outputs/transcription_with_timestamps.srt
JSON: /Users/tamdod/Documents/VS_Code_Projects/Whisper STT Implementation/transcription_outputs/transcription_with_timestamps.json


In [12]:
# Check that all expected export files exist.
export_files = [
    txt_path,
    srt_path,
    json_path,
]

for export_file in export_files:
    if not export_file.exists():
        raise FileNotFoundError(f"Missing export: {export_file}")

    print(
        f"{export_file.name}: "
        f"{export_file.stat().st_size / 1024:.2f} KB"
    )


# Read the JSON export back into Python to confirm it is valid.
with json_path.open("r", encoding="utf-8") as json_file:
    saved_json_data = json.load(json_file)


# Confirm that the JSON contains timestamped segments.
saved_segments = saved_json_data.get("segments", [])

if not saved_segments:
    raise ValueError("The JSON export contains no segments.")


# Display a sample of the first subtitle entry.
with srt_path.open("r", encoding="utf-8") as subtitle_file:
    first_srt_lines = subtitle_file.read().splitlines()[:5]


print("\nExport validation successful.")
print(f"JSON segments: {len(saved_segments)}")
print("\nFirst SRT entry:")
print("\n".join(first_srt_lines))

transcription_with_timestamps.txt: 1.77 KB
transcription_with_timestamps.srt: 2.03 KB
transcription_with_timestamps.json: 5.02 KB

Export validation successful.
JSON segments: 25

First SRT entry:
1
00:00:00,000 --> 00:00:09,000
CEDAS has entered Customs and the next step is to prepare a Mexican Customs entry.

2
